# Experiment 1: Pipelined KV-Cache Attention

Compare the cost (cycles and energy) of KV-cache attention with and without pipelining.

Three workloads are evaluated against the TPU v4i architecture:
- **Baseline** (`gpt3_175B_kv_cache.yaml`): full context processed in one pass
- **2-chunk pipeline** (`gpt3_175B_kv_cache_pipeline2.yaml`): context split into 2 chunks, accumulated via vector unit
- **8-chunk pipeline** (`gpt3_175B_kv_cache_pipeline8.yaml`): context split into 8 chunks, accumulated via vector unit

Each workload is mapped automatically with AccelForge's mapper.

In [26]:
from accelforge import Spec, examples
from pathlib import Path

In [27]:
def get_cycles(result):
    return float(result.latency())

def get_energy(result):
    return float(result.energy())

def get_component_energy(result, component):
    energy = result.energy(per_component=True)
    return float(energy.get(component, 0))

def get_component_cyles(result, component):
    latency = result.energy(per_component=True)
    return float(latency.get(component, 0))

## P=2 set up

In [28]:
def get_results(qk_results, sm_results, av_results, acc_results):
    # Total Pipeline Cycles
    t0_cycles = get_cycles(qk_results)
    t1_cycles = max(get_cycles(qk_results), get_cycles(sm_results))
    t2_cycles = max(get_cycles(av_results), get_cycles(sm_results))
    t3_cycles = get_cycles(av_results)
    t4_cycles = get_cycles(acc_results)
    
    print("Total Pipeline Cycles: ", t0_cycles+t1_cycles+t2_cycles+t3_cycles+t4_cycles)
    
    # Total Pipeline Energy
    t0_energy = get_energy(qk_results)
    t1_energy = max(get_energy(qk_results), get_energy(sm_results))
    t2_energy = max(get_energy(av_results), get_energy(sm_results))
    t3_energy = get_energy(av_results)
    t4_energy = get_energy(acc_results)
    
    print("Total Pipeline Energy: ", t0_energy+t1_energy+t2_energy+t3_energy+t4_energy)

## 25% GLB Capacity

In [29]:
qk_spec_25 = Spec.from_yaml(
    "../arches/25_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_25 = qk_spec_25.map_workload_to_arch()

sm_spec_25 = Spec.from_yaml(
    "../arches/25_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_25 = sm_spec_25.map_workload_to_arch()

av_spec_25 = Spec.from_yaml(
    "../arches/25_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_25 = av_spec_25.map_workload_to_arch()

acc_spec_25 = Spec.from_yaml(
    "../arches/25_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_25 = acc_spec_25.map_workload_to_arch()

get_results(qk_results_25, sm_results_25, av_results_25, acc_results_25)

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 14.77it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 9it [00:00, 89.17it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 19it [00:01, 14.39it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:01, 25.04it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.35s/it]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 311.03it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 788.40it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.91e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1121.77it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1158.01it/s]


Dirty joining mapping(s) valid & optimal! Returning...
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732


## 50% GLB Capacity

In [30]:
qk_spec_50 = Spec.from_yaml(
    "../arches/50_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_50 = qk_spec_50.map_workload_to_arch()

sm_spec_50 = Spec.from_yaml(
    "../arches/50_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_50 = sm_spec_50.map_workload_to_arch()

av_spec_50 = Spec.from_yaml(
    "../arches/50_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_50 = av_spec_50.map_workload_to_arch()

acc_spec_50 = Spec.from_yaml(
    "../arches/50_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_50 = acc_spec_50.map_workload_to_arch()

get_results(qk_results_50, sm_results_50, av_results_50, acc_results_50)

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 28.17it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 125.22it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 112.31it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.91it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 350.64it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 939.37it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.91e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1196.66it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7157.52it/s]


Dirty joining mapping(s) valid & optimal! Returning...
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732


## 100% GLB Capacity

In [31]:
qk_spec_100 = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_100 = qk_spec_100.map_workload_to_arch()

sm_spec_100 = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_100 = sm_spec_100.map_workload_to_arch()

av_spec_100 = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_100 = av_spec_100.map_workload_to_arch()

acc_spec_100 = Spec.from_yaml(
    "../arches/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_100 = acc_spec_100.map_workload_to_arch()

get_results(qk_results_100, sm_results_100, av_results_100, acc_results_100)

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 25.69it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 122.52it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 114.74it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.03it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 336.16it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 855.81it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.91e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 653.11it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7898.88it/s]


Dirty joining mapping(s) valid & optimal! Returning...
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732


## 200% GLB Capacity

In [32]:
qk_spec_200 = Spec.from_yaml(
    "../arches/200_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_200 = qk_spec_200.map_workload_to_arch()

sm_spec_200 = Spec.from_yaml(
    "../arches/200_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_200 = sm_spec_200.map_workload_to_arch()

av_spec_200 = Spec.from_yaml(
    "../arches/200_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_200 = av_spec_200.map_workload_to_arch()

acc_spec_200 = Spec.from_yaml(
    "../arches/200_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_200 = acc_spec_200.map_workload_to_arch()

get_results(qk_results_200, sm_results_200, av_results_200, acc_results_200)

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 24.19it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 14it [00:00, 131.99it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 120.64it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.16it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 438.18it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1172.58it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.91e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 780.34it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5190.97it/s]


Dirty joining mapping(s) valid & optimal! Returning...
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732


## Comparisons

In [33]:
print('\n25 results:')
get_results(qk_results_25, sm_results_25, av_results_25, acc_results_25)

print('\n50 results:')
get_results(qk_results_50, sm_results_50, av_results_50, acc_results_50)

print('\n100 results:')
get_results(qk_results_100, sm_results_100, av_results_100, acc_results_100)

print('\n200 results:')
get_results(qk_results_200, sm_results_200, av_results_200, acc_results_200)



25 results:
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732

50 results:
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732

100 results:
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732

200 results:
Total Pipeline Cycles:  0.0004563174097711453
Total Pipeline Energy:  0.017588161050237732


In [34]:

# Check GLB sizes and see what memory stats the result exposes
glb_25  = 1024*1024*128*2
glb_50  = 1024*1024*128*4
glb_100 = 1024*1024*128*8
glb_200 = 1024*1024*128*16

print(f"GLB sizes (bytes): 25%={glb_25:,}  50%={glb_50:,}  100%={glb_100:,}  200%={glb_200:,}")

# Inspect what the result object exposes
print("\nResult attributes:", [a for a in dir(qk_results_25) if not a.startswith('_')])


GLB sizes (bytes): 25%=268,435,456  50%=536,870,912  100%=1,073,741,824  200%=2,147,483,648

Result attributes: ['access', 'actions', 'columns', 'data', 'drop', 'drop_components', 'drop_components_with_zero_energy_and_latency', 'drop_zeros', 'einsum_names', 'energy', 'evaluated_specs', 'flattened_arches', 'items', 'latency', 'mapping', 'n_computes', 'per_compute', 'per_tensor_size', 'render', 'resource_usage', 'spec', 'sum', 'to_dict', 'total_mappings', 'valid_mappings']


In [35]:
# Compare max tensor footprint vs GLB sizes
M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Largest tensor in QK stage: K_1[M_CHUNK, H, E]
k1_bytes = M_CHUNK * H * E * bytes_per_val
print(f"K_1 size: {k1_bytes / 1e6:.1f} MB")

glbs = {
    "25%":  1024*1024*128*2,
    "50%":  1024*1024*128*4,
    "100%": 1024*1024*128*8,
    "200%": 1024*1024*128*16,
}
for name, size in glbs.items():
    print(f"GLB {name}: {size/1e6:.0f} MB  — K_1 fits: {k1_bytes < size}")

K_1 size: 67.1 MB
GLB 25%: 268 MB  — K_1 fits: True
GLB 50%: 537 MB  — K_1 fits: True
GLB 100%: 1074 MB  — K_1 fits: True
GLB 200%: 2147 MB  — K_1 fits: True
